# LUNA CLOUD — Colab Runtime

This notebook connects a Colab GPU runtime outward to your LUNA CLOUD backend.

## BEFORE running any cells

### 1. Set `COMPUTE_PROVIDER=colab` on Render
Go to **Render Dashboard → Settings → Environment** and confirm:
```
COMPUTE_PROVIDER=colab
STREAMING_PROVIDER=webrtc
```
Then click **Restart service** and wait 3 minutes.

### 2. Set Colab Secret
Colab sidebar → 🔒 Secrets → Add new secret:
- **Name:** `LUNA_RUNTIME_AUTH`
- **Value:** (paste from **Render Dashboard → Environment → `RUNTIME_AUTH_SECRET`** value)

### 3. Run cells in order below

In [ ]:
import os
import sys

# ============================================================
# CONFIGURATION
# ============================================================

LUNA_BACKEND_WS = os.environ.get("LUNA_BACKEND_WS", "wss://kyro-cloud-3fp0.onrender.com/agent")

# Runtime auth secret — must match Render's RUNTIME_AUTH_SECRET
# CRITICAL: Create this secret in Colab FIRST!
# Colab sidebar → 🔒 Secrets → Add: LUNA_RUNTIME_AUTH = (paste Render RUNTIME_AUTH_SECRET)
try:
    from google.colab import userdata
    _secret = userdata.get("LUNA_RUNTIME_AUTH")
except Exception:
    _secret = None

RUNTIME_AUTH_SECRET = os.environ.get("RUNTIME_AUTH_SECRET", _secret or "runtime-change-me")
REPO = os.environ.get("LUNA_REPO", "https://github.com/harshpreetsaini/kyo-cloud.git")

os.environ["LUNA_BACKEND_WS"] = LUNA_BACKEND_WS
os.environ["RUNTIME_AUTH_SECRET"] = RUNTIME_AUTH_SECRET

# ============================================================
# DIAGNOSTICS — run this FIRST to catch setup errors
# ============================================================

print("=== CONFIGURATION CHECK ===")
print("Backend WS:       ", LUNA_BACKEND_WS)
print("Auth secret:      ", "LOADED ✅" if (_secret or RUNTIME_AUTH_SECRET != "runtime-change-me") else "NOT SET ❌")
print("Auth value:       ", repr(RUNTIME_AUTH_SECRET[:8]) + "..." if RUNTIME_AUTH_SECRET != "runtime-change-me" else "UNKNOWN")

# Check Colab secret
has_secret = False
try:
    secrets = userdata._get_user_secrets()
    has_secret = "LUNA_RUNTIME_AUTH" in secrets
except Exception:
    pass
print("Colab secret:     ", "FOUND ✅" if has_secret else "NOT FOUND ❌")

if not has_secret:
    print("\n" + "="*50)
    print("❌ FIX THIS FIRST:")
    print("="*50)
    print("1. Click the 🔒 Secrets icon on the LEFT sidebar")
    print("2. Click 'Add new secret'")
    print("3. Name: LUNA_RUNTIME_AUTH")
    print("4. Value: (paste Render RUNTIME_AUTH_SECRET)")
    print("   → Render Dashboard → Settings → Environment → RUNTIME_AUTH_SECRET")
    print("5. Re-run this cell after saving the secret.")
    print("="*50)


In [ ]:
import subprocess

# ============================================================
# CLONE REPO (always get latest code)
# ============================================================

def run(cmd, label=""):
    print(f"\n=== {label} ===")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout.strip()
    err = r.stderr.strip()
    if out: print(out[:3000])
    if err and "W:" not in err: print("STDERR:", err[:500])
    return r.returncode

run("git clone https://github.com/harshpreetsaini/kyro-cloud.git /content/luna-cloud 2>/dev/null || (cd /content/luna-cloud && git pull)", "CLONE")

os.chdir("/content/luna-cloud")
print("\nWorking dir:", os.getcwd())


In [ ]:
import subprocess, time, os

# ============================================================
# BOOTSTRAP
# ============================================================

print("Running bootstrap...")
print("This installs: xfce4, tigervnc, dbus-x11, openbox + Python deps")
print("It will also START the agent (connect to Render backend).")
print()

r = subprocess.run(
    "sudo -E bash runtime-agent/bootstrap/bootstrap.sh",
    shell=True, capture_output=True, text=True
)

out = r.stdout.strip()
print(out)
if r.returncode != 0:
    print("BOOTSTRAP EXIT CODE:", r.returncode)
    print(r.stderr[-1000:])
    print("\n❌ Bootstrap failed! Check errors above.")
else:
    print("\n✅ Bootstrap completed.")

# Verify agent started
time.sleep(3)
print("\n=== AGENT STATUS ===")
r2 = subprocess.run("ps aux | grep '[p]ython3 main.py'", shell=True, capture_output=True, text=True)
print(r2.stdout or "⚠️  Agent not running!")

print("\n=== NEXT STEPS ===")
print("1. If agent is running → go to Vercel, hard-refresh (Ctrl+Shift+R), click Start Session")
print("2. If agent NOT running → check errors above and re-run this cell")
print("3. On Vercel: Ctrl+Shift+R → Start Session → status should go Online")
